# Smart Bus Safety Model: Logic & Implementation

This notebook demonstrates the underlying logic of the **Rollover Risk & Stopping Distance Prediction Model** used in the Smart Bus System.

## Objective
To predict two critical safety metrics in real-time:
1. **Rollover Risk Score (0-1)**: Based on lateral acceleration and bus stability.
2. **Stopping Distance (m)**: Based on speed, friction, and reaction time.

## Methodology: Digital Twin Approach
Since real-world rollover data is dangerous to collect, we use a **Physics-based Digital Twin** to generate ground truth data, which acts as the teacher for our Machine Learning model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

## 1. Physics Constants
These constants define the physical properties of the **Ashok Leyland Viking** bus (localized for Sri Lankan context).

In [ ]:
# Constants
MASS_BUS = 9500     # kg (Chassis + Body)
MASS_PAX = 62       # kg (Avg adult weight)
H_EMPTY = 1.15      # m  (CoG height empty)
H_SEAT = 1.35       # m  (CoG seated)
H_STAND = 2.15      # m  (CoG standing)
TRACK_WIDTH = 1.95  # m
G = 9.81            # m/s²
FRICTION_DRY = 0.65
FRICTION_WET = 0.35
REACTION_TIME = 1.8 # s

## 2. Synthetic Data Generation (The "Digital Twin")
We simulate thousands of driving scenarios. For each scenario, we calculate the **Ground Truth** using physics formulas:

- **Lateral Acceleration**: $a_{lat} = v^2 / r$
- **Static Stability Factor (SSF)**: $T / (2 \times h_{cog})$
- **Risk Score**: $a_{lat} / SSF$

### New Feature: Smooth Warnings
We recently added `dist_to_curve_m`. The risk score is "decayed" (smoothed) based on how far the bus is from the curve.
$$ Risk_{smooth} = \frac{Risk_{raw}}{1 + 0.05 \times Distance} $$

In [ ]:
def generate_synthetic_data(n_samples=5000):
    # Random Inputs
    n_seated = np.random.randint(0, 55, n_samples)
    n_standing = np.random.randint(0, 40, n_samples)
    speed_kmh = np.random.uniform(20, 80, n_samples)
    radius_m = np.random.uniform(10, 200, n_samples) # Curve radius
    is_wet = np.random.choice([0, 1], n_samples) # 0=Dry, 1=Wet
    gradient_deg = np.random.uniform(-5, 5, n_samples) # Slope
    dist_to_curve_m = np.random.uniform(0, 150, n_samples) # Distance to curve
    
    data = []
    
    for i in range(n_samples):
        # 1. Mass & Center of Gravity (CoG)
        m_seated = n_seated[i] * MASS_PAX
        m_standing = n_standing[i] * MASS_PAX
        m_total = MASS_BUS + m_seated + m_standing
        
        h_cog = ((MASS_BUS * H_EMPTY) + (m_seated * H_SEAT) + (m_standing * H_STAND)) / m_total
        
        # 2. Rollover Threshold (SSF)
        ssf = TRACK_WIDTH / (2 * h_cog)
        
        # 3. Lateral Acceleration
        v_ms = speed_kmh[i] / 3.6
        lat_accel = (v_ms ** 2) / radius_m[i]
        lat_accel_g = lat_accel / G
        
        # 4. Risk Score (Raw)
        raw_risk_score = lat_accel_g / ssf
        
        # SMOOTHING LOGIC: Decay risk as distance increases
        decay_factor = 1.0 / (1.0 + 0.05 * dist_to_curve_m[i])
        smooth_risk_score = raw_risk_score * decay_factor
        
        # 5. Stopping Distance
        friction = FRICTION_WET if is_wet[i] else FRICTION_DRY
        eff_friction = max(0.1, friction + np.tan(np.radians(gradient_deg[i])))
        
        d_reaction = v_ms * REACTION_TIME
        d_braking = (v_ms ** 2) / (2 * G * eff_friction)
        total_stopping_dist = d_reaction + d_braking
        
        data.append({
            'n_seated': n_seated[i],
            'n_standing': n_standing[i],
            'speed_kmh': speed_kmh[i],
            'radius_m': radius_m[i],
            'is_wet': is_wet[i],
            'gradient_deg': gradient_deg[i],
            'dist_to_curve_m': dist_to_curve_m[i],
            'risk_score': smooth_risk_score,
            'stopping_dist': total_stopping_dist
        })
        
    return pd.DataFrame(data)

# Generate Dataset
df = generate_synthetic_data(5000)
print(f"Generated {len(df)} samples.")
df.head()

## 3. Model Training
We use a **Multi-Output Random Forest Regressor** to predict both targets simultaneously.

In [ ]:
# Prepare Data
X = df[['n_seated', 'n_standing', 'speed_kmh', 'radius_m', 'is_wet', 'gradient_deg', 'dist_to_curve_m']]
y = df[['risk_score', 'stopping_dist']]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Result: Model R2 Score (Accuracy): {r2:.4f}")

## 4. Feature Importance Analysis
Let's see which factors contribute most to the risk. We expect **Speed** and **Radius** to be dominant for Rollover Risk.

In [ ]:
target_names = ['Rollover Risk', 'Stopping Distance']
features = X.columns

for i, target_name in enumerate(target_names):
    estimator = model.estimators_[i]
    importances = estimator.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    plt.title(f"Feature Importance for {target_name}")
    plt.bar(range(X.shape[1]), importances[indices], align='center')
    plt.xticks(range(X.shape[1]), [features[f] for f in indices], rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 3 Factors for {target_name}:")
    for f in range(3):
        print(f"{f+1}. {features[indices[f]]} ({importances[indices[f]]:.2f})")

## 5. Live Prediction Demo
Simulate a live API call to test the model.

In [ ]:
def predict_live(speed, radius, seated, standing, dist_to_curve=0):
    input_data = pd.DataFrame([{
        'n_seated': seated,
        'n_standing': standing,
        'speed_kmh': speed,
        'radius_m': radius,
        'is_wet': 0,
        'gradient_deg': 0,
        'dist_to_curve_m': dist_to_curve
    }])
    
    pred = model.predict(input_data)[0]
    risk = pred[0]
    
    status = "SAFE"
    if risk > 0.5: status = "WARNING"
    if risk > 0.7: status = "CRITICAL"
    
    print(f"Scenario: {speed}km/h on {radius}m curve (Dist: {dist_to_curve}m)")
    print(f"Predicted Risk: {risk:.4f} [{status}]")
    print("-" * 30)

# Test Cases
predict_live(speed=40, radius=30, seated=40, standing=10, dist_to_curve=0)  # Sharp turn, close
predict_live(speed=60, radius=30, seated=40, standing=10, dist_to_curve=100) # Sharp turn, far away (Should be lower risk)
predict_live(speed=80, radius=30, seated=40, standing=10, dist_to_curve=0)  # High speed, sharp turn (Critical)